# 00 — Authoring the qrels

**qrels are the ground truth**: for each query, which documents answer it, which merely relate
to it, and which do not.

Without them, every metric in this project (MRR, nDCG@k) is just a number. This notebook gets
reused: the first 12 queries were written here, and it is the place to extend the set to 40-50.

## Grading scale (applied consistently to every query)

| Score | Meaning | Question to ask |
|---|---|---|
| **2** | The document **answers** the query directly | "After reading this, does the asker know the answer?" |
| **1** | **Partially relevant** — same topic, no direct answer | "It is about something else, but it mentions the concept" |
| **0** | Not relevant (only graded after actually reading the document) | "This does not help with the query at all" |

## Three steps per query

1. `search("...")` — the full question works; the tool segments words itself
2. `show("<doc_id>")` — read a candidate in full
3. `judge(...)` — record the grades

Finish with `save()` → writes `eval/qrels.csv`.


In [1]:
# ── Setup: load corpus + index (run once, ~5-10 s) ─────────────
import json, sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "scripts"))

from explore import load_docs, build_index, search as _search   # same code the CLI uses

QRELS = ROOT / "eval" / "qrels.csv"
_docs, SOURCE = load_docs()
INDEX = build_index(_docs)
docs = pd.DataFrame(_docs)
print(f"corpus: {len(docs)} articles · {docs['chars'].sum()/1e6:.2f} MB · source: {SOURCE}")


corpus: 179 articles · 2.47 MB · source: curated (data\processed\curated.jsonl)


In [2]:
# ── Grading tools ─────────────────────────────────────────────
def search(query: str, k: int = 8):
    """Rank documents for a query — a full question or a few keywords both work."""
    rows = _search(query, _docs, k=k, index=INDEX)
    if not rows:
        return f"no documents found for '{query}'"
    return pd.DataFrame(rows)[["doc_id", "title", "chars", "matched", "score", "snippet"]]

def show(doc_id, limit: int = 2000):
    """Print a document in full (or its first `limit` characters) so it can be graded."""
    row = docs[docs["doc_id"] == str(doc_id)]
    if row.empty:
        print(f"no such doc_id: {doc_id}"); return
    r = row.iloc[0]
    print(f"# {r['title']}\n{r['url']}\n({r['chars']:,} chars)\n")
    print(r["text"][:limit] + ("\n\n[... truncated — raise limit to read more]" if r["chars"] > limit else ""))

# grading log, reloaded if grading was already started in an earlier session
if QRELS.exists():
    jud = pd.read_csv(QRELS, dtype={"doc_id": str}).dropna(subset=["relevance"]).to_dict("records")
    print(f"loaded {len(jud)} existing rows from {QRELS.relative_to(ROOT)}")
else:
    jud = []
    print("no qrels file yet — save() creates it")

def judge(query_id, query, grades, query_class):
    """Record grades for one query.

    Safe to re-run: a query_id can never change its question text, and
    (query_id, doc_id) pairs that were already graded are skipped. Losing that
    guard once duplicated 12 rows in qrels.csv and a typo in a query string
    silently shadowed the correct one (dict() keeps the last write).
    """
    assert query_class in {"literal", "paraphrase", "multi-doc"}, "query_class: literal / paraphrase / multi-doc"
    existing = {r["query_id"]: r["query"] for r in jud}
    if query_id in existing and existing[query_id] != query:
        raise ValueError(f"{query_id} already has a different query: '{existing[query_id]}' != '{query}'")
    already = {(r["query_id"], r["doc_id"]) for r in jud}
    added = 0
    for did, rel in grades.items():
        assert str(did) in set(docs["doc_id"]), f"doc_id {did} is not in the corpus"
        assert rel in (0, 1, 2), "relevance must be 0 / 1 / 2"
        if (query_id, str(did)) in already:
            continue
        jud.append({"query_id": query_id, "query": query, "doc_id": str(did),
                    "relevance": rel, "query_class": query_class})
        added += 1
    print(f"{query_id}: +{added} new rows -> {len(jud)} total")

def progress():
    """Print what has been graded so far, by class."""
    if not jud:
        print("nothing graded yet."); return
    df = pd.DataFrame(jud)
    print(df.groupby(["query_id", "query_class"]).agg(docs=("doc_id", "count")).reset_index().to_string(index=False))
    print("\nby class:", df.groupby("query_class")["query_id"].nunique().to_dict())
    print(f"total: {df['query_id'].nunique()} queries · {len(df)} rows")

def save():
    """Write eval/qrels.csv, deduplicated on (query_id, doc_id)."""
    if not jud:
        print("nothing graded — not overwriting the qrels file."); return
    df = pd.DataFrame(jud).drop_duplicates(subset=["query_id", "doc_id"], keep="first")[
        ["query_id", "query", "doc_id", "relevance", "query_class"]]
    QRELS.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(QRELS, index=False)
    print(f"saved {len(df)} rows -> {QRELS.relative_to(ROOT)}")

print("ready: search() · show() · judge() · progress() · save()")


loaded 31 existing rows from eval\qrels.csv
ready: search() · show() · judge() · progress() · save()


## Walkthrough — one query, end to end

The full question can be typed as-is (the tool segments words and drops stopwords), or a few
keywords. The tool only **finds candidates**; it does not grade. It is deliberately simple
(IDF + length normalisation) — the BM25 built in notebook 02 is strictly better, and wherever
its ranking looks wrong that is useful evidence for the comparison later.

A candidate whose title matches the topic is almost always a 2. A document that only *mentions*
the concept has to be read before it can be graded.


In [3]:
search("Học có giám sát là gì?")


,doc_id,title,chars,matched,score,snippet
0,19895408,Học tự giám sát,8010,3,210.61,Học tự giám sát (self-supervised learning hay ...
1,19893713,Học đặc trưng,24688,3,170.05,"Trong học máy, học đặc trưng (feature learning..."
2,3339820,Học sâu,57004,3,137.83,"Học sâu (tiếng Anh: deep learning, còn gọi là ..."
3,24345,Học không có giám sát,1866,3,129.94,Học không có giám sát (tiếng Anh: unsupervised...
4,20014464,Thuật ngữ trí tuệ nhân tạo,42756,3,119.03,"... đến ngành trí tuệ nhân tạo (AI), các phân ..."
5,19950271,Học vấn về AI,8233,3,107.72,Học vấn về AI hay học vấn về trí tuệ nhân tạo ...
6,24278,Học máy,19947,3,97.68,Học máy (tiếng Anh: Machine learning) là một l...
7,20012838,Tốc độ học,7071,3,91.16,"Tốc độ học (tiếng Anh: learning rate, còn được..."


In [4]:
show("19895408", limit=1200)


# Học tự giám sát
https://vi.wikipedia.org/wiki/H%E1%BB%8Dc_t%E1%BB%B1_gi%C3%A1m_s%C3%A1t
(8,010 chars)

Học tự giám sát (self-supervised learning hay SSL) là một kỹ thuật trong học máy mà trong đó, một mô hình được huấn luyện trên một tác vụ bằng cách sử dụng chính dữ liệu để tạo ra các tín hiệu giám sát, thay vì dựa vào các nhãn bên ngoài do con người cung cấp. Trong bối cảnh của mạng thần kinh nhân tạo, học tự giám sát hướng tới việc khai thác các cấu trúc hoặc mối quan hệ nội tại trong dữ liệu đầu vào để tạo ra các tín hiệu huấn luyện có ý nghĩa. Các nhiệm vụ trong SSL được thiết kế sao cho việc giải quyết yêu cầu nắm bắt được các đặc điểm hoặc mối quan hệ quan trọng trong dữ liệu. Dữ liệu đầu vào thường được tăng cường hoặc biến đổi để tạo ra các cặp mẫu liên quan. Một mẫu sẽ đóng vai trò làm đầu vào, và mẫu còn lại được dùng để hình thành tín hiệu giám sát. Việc tăng cường này có thể bao gồm thêm nhiễu, cắt, xoay, hoặc các biến đổi khác. Học tự giám sát mô phỏng cách con người họ

In [5]:
# Grading is one call per query: a {doc_id: relevance} mapping plus the query class.
# This sample is kept as the template for every query that follows.

# judge("q001", "Học có giám sát là gì?", {
#     "24329":    2,   # the article on the topic — defines it directly
#     "19895408": 1,   # self-supervised learning — adjacent topic, only referenced
#     "20014464": 1,   # glossary entry — one defining line, no explanation
# }, "literal")

progress()


query_id query_class  docs
    q001     literal     3
    q002     literal     3
    q003     literal     3
    q004     literal     3
    q005  paraphrase     3
    q006  paraphrase     1
    q007  paraphrase     2
    q008  paraphrase     2
    q009   multi-doc     3
    q010   multi-doc     2
    q011   multi-doc     3
    q012   multi-doc     3

by class: {'literal': 4, 'multi-doc': 4, 'paraphrase': 4}
total: 12 queries · 31 rows


## Batch 1 — the 12 queries

The set deliberately mixes three kinds of question, because a retriever that wins on one kind
can lose badly on another:

- **literal** (4) — uses the terms the articles use (*"Mạng nơ-ron tích chập là gì?"*)
- **paraphrase** (4) — no term overlap with the answer (*"Máy tính học từ ví dụ có nhãn bằng cách nào?"* rather than *"học có giám sát"*)
- **multi-doc** (4) — needs at least two documents (*"So sánh học có giám sát và học không có giám sát"*)

Topics were picked from subjects already studied, which makes the relevance judgements far more
confident than grading unfamiliar material.

`eval/qrels.csv` is the authoritative record for all 12 queries; the cells below grade the first
four end to end, and the same tool was used for the rest.


In [6]:
# ── q001 ──────────────────────────────────────────────────────
search("Học không có giám sát là gì?")


,doc_id,title,chars,matched,score,snippet
0,19895408,Học tự giám sát,8010,3,210.61,Học tự giám sát (self-supervised learning hay ...
1,19893713,Học đặc trưng,24688,3,170.05,"Trong học máy, học đặc trưng (feature learning..."
2,3339820,Học sâu,57004,3,137.83,"Học sâu (tiếng Anh: deep learning, còn gọi là ..."
3,24345,Học không có giám sát,1866,3,129.94,Học không có giám sát (tiếng Anh: unsupervised...
4,20014464,Thuật ngữ trí tuệ nhân tạo,42756,3,119.03,"... đến ngành trí tuệ nhân tạo (AI), các phân ..."
5,19950271,Học vấn về AI,8233,3,107.72,Học vấn về AI hay học vấn về trí tuệ nhân tạo ...
6,24278,Học máy,19947,3,97.68,Học máy (tiếng Anh: Machine learning) là một l...
7,20012838,Tốc độ học,7071,3,91.16,"Tốc độ học (tiếng Anh: learning rate, còn được..."


In [7]:
show("20012838")


# Tốc độ học
https://vi.wikipedia.org/wiki/T%E1%BB%91c_%C4%91%E1%BB%99_h%E1%BB%8Dc
(7,071 chars)

Tốc độ học (tiếng Anh: learning rate, còn được dịch là Tỉ lệ học) là một siêu tham số (hyperparameter) trong các thuật toán tối ưu hóa như suy giảm độ dốc, giúp xác định kích thước mỗi bước cập nhật khi tham số của mô hình di chuyển về phía cực tiểu của hàm mất mát. 
Tốc độ học quá cao có thể khiến quá trình học "nhảy qua" cực tiểu cần tìm; ngược lại, tốc độ học quá thấp khiến quá trình hội tụ chậm hoặc mắc kẹt tại một điểm cực tiểu địa phương không mong muốn. Để tránh các vấn đề trên, tốc độ học có thể được thay đổi trong quá trình huấn luyện theo một lịch trình định trước (learning rate schedule) hoặc bằng một tốc độ học thích ứng (adaptive learning rate). 
Trong các tài liệu về điều khiển thích nghi (adaptive control) và xấp xỉ ngẫu nhiên, đại lượng tương tự còn được gọi là hệ số khuếch đại (gain).


== Thay đổi tốc độ học trong quá trình huấn luyện ==
Tốc độ học khởi tạo có thể giữ một

In [8]:
show("24345")


# Học không có giám sát
https://vi.wikipedia.org/wiki/H%E1%BB%8Dc_kh%C3%B4ng_c%C3%B3_gi%C3%A1m_s%C3%A1t
(1,866 chars)

Học không có giám sát (tiếng Anh: unsupervised learning) là một phương pháp của ngành học máy nhằm tìm ra một mô hình mà phù hợp với các quan sát. Nó khác biệt với học có giám sát ở chỗ là đầu ra đúng tương ứng cho mỗi đầu vào là không biết trước. Trong học không có giám sát, một tập dữ liệu đầu vào được thu thập. Học không có giám sát thường đối xử với các đối tượng đầu vào như là một tập các biến ngẫu nhiên. Sau đó, một mô hình mật độ kết hợp sẽ được xây dựng cho tập dữ liệu đó.
Học không có giám sát có thể được dùng kết hợp với suy luận Bayes để cho ra xác suất có điều kiện (nghĩa là học có giám sát) cho bất kì biến ngẫu nhiên nào khi biết trước các biến khác.
Học không có giám sát cũng hữu ích cho việc nén dữ liệu: về cơ bản, mọi giải thuật nén dữ liệu hoặc là dựa vào một phân bố xác suất trên một tập đầu vào một cách tường minh hay không tường minh.
Một dạng khác 

In [9]:
show("19893713")


# Học đặc trưng
https://vi.wikipedia.org/wiki/H%E1%BB%8Dc_%C4%91%E1%BA%B7c_tr%C6%B0ng
(24,688 chars)

Trong học máy, học đặc trưng (feature learning) hay học biểu diễn (representation learning) là tập hợp các kỹ thuật cho phép hệ thống tự động khám phá các biểu diễn cần thiết cho việc phát hiện hoặc phân loại đặc trưng từ dữ liệu thô. Điều này thay thế cho việc xây dựng đặc trưng (feature engineering) thủ công và cho phép máy học cả đặc trưng lẫn thực hiện nhiệm vụ cụ thể.
Học đặc trưng được thúc đẩy bởi thực tế rằng các bài toán trong học máy như phân loại thường yêu cầu đầu vào dễ dàng và thuận tiện cho việc xử lý toán học và tính toán. Tuy nhiên, dữ liệu thực tế như hình ảnh, video và dữ liệu từ cảm biến không thể dễ dàng xác định các đặc trưng cụ thể thông qua các thuật toán rõ ràng. Một phương pháp thay thế là khám phá các đặc trưng hoặc biểu diễn này thông qua quá trình quan sát, mà không phụ thuộc vào các thuật toán cụ thể.
Học đặc trưng có thể là có giám sát, không giám sát hoặ

In [10]:
judge("q001", "Học không có giám sát là gì?", {
    "24345": 2, "19893713": 1, "20012838": 0
}, "literal")

progress()


q001: +0 new rows -> 31 total
query_id query_class  docs
    q001     literal     3
    q002     literal     3
    q003     literal     3
    q004     literal     3
    q005  paraphrase     3
    q006  paraphrase     1
    q007  paraphrase     2
    q008  paraphrase     2
    q009   multi-doc     3
    q010   multi-doc     2
    q011   multi-doc     3
    q012   multi-doc     3

by class: {'literal': 4, 'multi-doc': 4, 'paraphrase': 4}
total: 12 queries · 31 rows


In [11]:
# ── q002 ──────────────────────────────────────────────────────
search("Phân loại nhị phân là gì?")


,doc_id,title,chars,matched,score,snippet
0,16581586,Phân kỳ Kullback–Leibler,74966,4,269.28,"Phân kỳ Kullback–Leibler (viết tắt KL, còn gọi..."
1,20023680,Phân loại đa lớp,24460,4,200.97,"Trong học máy và phân loại thống kê, phân loại..."
2,60027,Tổng hợp giọng nói,49950,4,151.60,...hệ thống tổng hợp giọng nói nào cũng có đầu...
3,30712,Phân loại nhị phân,4112,4,139.46,Phân loại nhị phân (tiếng Anh: Binary classifi...
4,20014464,Thuật ngữ trí tuệ nhân tạo,42756,4,132.30,...và khái niệm liên quan đến ngành trí tuệ nh...
5,20015375,Entropy chéo,24899,4,121.52,...ng trong lý thuyết thông tin đo lường sự kh...
6,19893713,Học đặc trưng,24688,4,120.92,...ng khám phá các biểu diễn cần thiết cho việ...
7,19991608,Perceptron,25201,4,96.78,"..., perceptron là một thuật toán học có giám ..."


In [12]:
show("30712")


# Phân loại nhị phân
https://vi.wikipedia.org/wiki/Ph%C3%A2n_lo%E1%BA%A1i_nh%E1%BB%8B_ph%C3%A2n
(4,112 chars)

Phân loại nhị phân (tiếng Anh: Binary classification) là nhiệm vụ phận loại các phần tử của một tập hợp các đối tượng ra thành 2 nhóm dựa trên cơ sở là chúng có một thuộc tính nào đó hay không (hay còn gọi là tiêu chí). Một số nhiệm vụ phân loại nhị phân điển hình:

kiểm tra y khoa xem một bệnh nhân có bệnh nào đó hay không (thuộc tính để phân loại là căn bệnh đó)
quản lý chất lượng trong nhà máy, ví dụ: xác định xem một sản phẩm làm ra là đủ tốt để bán chưa, hay nên loại bỏ nó (thuộc tính để phân loại là tính đủ tốt)
xác định xem một trang hay một bài báo có nên nằm trong tập kết quả của một truy vấn hay không (thuộc tính là độ liên quan của bài báo - thường là sự hiện diện của một số từ nào đó trong bài báo đó)
Phân loại nói chung là một trong những vấn đề được nghiên cứu trong khoa học máy tính với mục đích học tự động các hệ thống phân loại. Một số phương pháp thích hợp ch

In [13]:
show("19893713")


# Học đặc trưng
https://vi.wikipedia.org/wiki/H%E1%BB%8Dc_%C4%91%E1%BA%B7c_tr%C6%B0ng
(24,688 chars)

Trong học máy, học đặc trưng (feature learning) hay học biểu diễn (representation learning) là tập hợp các kỹ thuật cho phép hệ thống tự động khám phá các biểu diễn cần thiết cho việc phát hiện hoặc phân loại đặc trưng từ dữ liệu thô. Điều này thay thế cho việc xây dựng đặc trưng (feature engineering) thủ công và cho phép máy học cả đặc trưng lẫn thực hiện nhiệm vụ cụ thể.
Học đặc trưng được thúc đẩy bởi thực tế rằng các bài toán trong học máy như phân loại thường yêu cầu đầu vào dễ dàng và thuận tiện cho việc xử lý toán học và tính toán. Tuy nhiên, dữ liệu thực tế như hình ảnh, video và dữ liệu từ cảm biến không thể dễ dàng xác định các đặc trưng cụ thể thông qua các thuật toán rõ ràng. Một phương pháp thay thế là khám phá các đặc trưng hoặc biểu diễn này thông qua quá trình quan sát, mà không phụ thuộc vào các thuật toán cụ thể.
Học đặc trưng có thể là có giám sát, không giám sát hoặ

In [14]:
show("20023680")


# Phân loại đa lớp
https://vi.wikipedia.org/wiki/Ph%C3%A2n_lo%E1%BA%A1i_%C4%91a_l%E1%BB%9Bp
(24,460 chars)

Trong học máy và phân loại thống kê, phân loại đa lớp (tiếng Anh: multiclass classification hay multinomial classification) là bài toán phân loại các đối tượng quan sát vào một trong ba hoặc nhiều lớp phân biệt (phân loại đối tượng vào đúng một trong hai lớp được gọi là phân loại nhị phân).
Ví dụ, việc xác định một hình ảnh chụp quả chuối, quả đào, quả cam hay quả táo là một bài toán phân loại đa lớp với 4 lớp xuất ra có thể xảy ra. Trong khi đó, việc xác định xem một hình ảnh có chứa quả táo hay không lại là bài toán phân loại nhị phân (với hai lớp: "có táo" và "không có táo").
Cần phân biệt rõ phân loại đa lớp với phân loại đa nhãn (multi-label classification) — nơi mỗi đối tượng có thể được gán đồng thời nhiều nhãn (ví dụ: một bức ảnh vừa có nhãn "quả táo" vừa có nhãn "quả cam") — và phân loại một lớp (one-class classification) nhằm phát hiện các điểm bất thường.


== Tổng qua

In [15]:
judge("q002", "Phân loại nhị phân là gì?", {
    "30712": 2, "20023680": 1, "19893713": 0
}, "literal")

progress()


q002: +0 new rows -> 31 total
query_id query_class  docs
    q001     literal     3
    q002     literal     3
    q003     literal     3
    q004     literal     3
    q005  paraphrase     3
    q006  paraphrase     1
    q007  paraphrase     2
    q008  paraphrase     2
    q009   multi-doc     3
    q010   multi-doc     2
    q011   multi-doc     3
    q012   multi-doc     3

by class: {'literal': 4, 'multi-doc': 4, 'paraphrase': 4}
total: 12 queries · 31 rows


In [16]:
# ── q003 ──────────────────────────────────────────────────────
search("Chưng cất tri thức là gì?")


,doc_id,title,chars,matched,score,snippet
0,25735,Lịch sử ngành trí tuệ nhân tạo,97944,4,148.06,Lịch sử ngành trí tuệ nhân tạo (AI) có những t...
1,19917007,Chưng cất tri thức,42272,4,140.78,"Trong học máy, chưng cất tri thức (knowledge d..."
2,19995683,Mùa đông AI,26218,4,91.50,...g AI (tiếng Anh: AI winter) là giai đoạn mà...
3,20017867,Thung lũng kỳ lạ,27899,4,89.59,...cho con người. Giả thuyết thung lũng kỳ lạ ...
4,20014464,Thuật ngữ trí tuệ nhân tạo,42756,4,75.99,...là danh sách các thuật ngữ và khái niệm liê...
5,20033670,Mô hình thị giác–ngôn ngữ–hành động,47226,4,65.20,"...model, viết tắt: VLA) là một loại mô hình h..."
6,19988121,Học liên kết,37355,4,62.55,"...c bộ của từng người dùng, nơi các phép tính..."
7,20035026,Thành viên:Qesefe/Quên thảm họa,25609,4,61.82,Quên thảm họa (tiếng Anh: catastrophic forgett...


In [17]:
show("19917007")


# Chưng cất tri thức
https://vi.wikipedia.org/wiki/Ch%C6%B0ng_c%E1%BA%A5t_tri_th%E1%BB%A9c
(42,272 chars)

Trong học máy, chưng cất tri thức (knowledge distillation) hay chưng cất mô hình (model distillation) là quá trình chuyển giao tri thức từ mô hình lớn sang mô hình nhỏ hơn. Dù các mô hình lớn (như mạng nơ-ron sâu hoặc tập hợp nhiều mô hình) có khả năng lưu trữ tri thức cao hơn, khả năng này có thể chưa được sử dụng hết. Chi phí tính toán để chạy mô hình vẫn cao ngay cả khi nó chỉ sử dụng một phần nhỏ tri thức. Chưng cất tri thức cho phép chuyển giao kiến thức từ mô hình lớn sang nhỏ mà không làm giảm hiệu quả. Các mô hình nhỏ tốn ít chi phí tính toán hơn, có thể triển khai trên thiết bị phần cứng yếu hơn (như điện thoại di động).
Khái niệm này khác với nén mô hình (model compression) - phương pháp giảm kích thước mô hình lớn mà không cần huấn luyện mô hình mới. Nén mô hình thường giữ nguyên kiến trúc và số lượng tham số, chỉ giảm số bit trên mỗi tham số.
Chưng cất tri thức đã được

In [18]:
show("25735")


# Lịch sử ngành trí tuệ nhân tạo
https://vi.wikipedia.org/wiki/L%E1%BB%8Bch_s%E1%BB%AD_ng%C3%A0nh_tr%C3%AD_tu%E1%BB%87_nh%C3%A2n_t%E1%BA%A1o
(97,944 chars)

Lịch sử ngành trí tuệ nhân tạo (AI) có những tiền đề từ thời cổ đại, khi con người kể các câu chuyện về những sinh vật nhân tạo và cỗ máy được ban cho trí thông minh. Qua nhiều thế kỷ, sự phát triển của logic, toán học và các phương pháp suy luận dựa trên quy tắc đã góp phần dẫn đến sự ra đời của máy tính điện tử vào thập niên 1940. Khi máy tính có thể thực hiện nhiều chuỗi thao tác khác nhau theo chương trình, các nhà khoa học bắt đầu nghiêm túc xem xét khả năng chế tạo một "bộ não điện tử", tức một hệ thống có thể mô phỏng một số năng lực trí tuệ của con người.
Trí tuệ nhân tạo chính thức hình thành như một lĩnh vực nghiên cứu tại một hội thảo tổ chức ở Đại học Dartmouth năm 1956. Tại đây, Allen Newell và Herbert A. Simon, với sự hỗ trợ của J. C. Shaw, trình bày Logic Theorist, một trong những chương trình AI đầu tiên. Newell về 

In [19]:
show("19989487")


# Machine unlearning
https://vi.wikipedia.org/wiki/Machine_unlearning
(18,327 chars)

Trong học máy, Machine unlearning (tạm dịch: Máy học cách quên) là một kỹ thuật giúp xóa kiến thức từ một phần dữ liệu đã học khỏi mô hình. Với sự phổ biến của các mô hình ngôn ngữ lớn như ChatGPT hay Gemini, nhiều người cho rằng chỉ cần yêu cầu mô hình "hãy quên đi" hay xóa đoạn chat là có thể khiến chúng "quên" những gì mình mong muốn. Tuy nhiên, trên thực tế, điều này không hề thay đổi tham số của mô hình mạng thần kinh nhân tạo, và chẳng mấy ảnh hưởng đến việc chúng có thực sự quên đi hay không.
Việc mô hình có thực sự "quên" đi hay không thường được định nghĩa theo "tiêu chuẩn vàng" là khiến mô hình phải học lại tất cả các kiến thức từ đầu theo đúng trình tự nó đã từng được huấn luyện, với điểm khác biệt duy nhất là không bao gồm phần dữ liệu mình muốn quên đi kia. Đây được gọi là phương pháp Học-lại-từ-đầu (Retrain-from-scratch), một giải pháp hết sức tốn kém nhưng đảm bảo được mô hình chắc chắn

In [20]:
judge("q003", "Chưng cất tri thức là gì?", {
    "19917007": 2, "19989487": 0, "25735": 0
}, "literal")

progress()


q003: +0 new rows -> 31 total
query_id query_class  docs
    q001     literal     3
    q002     literal     3
    q003     literal     3
    q004     literal     3
    q005  paraphrase     3
    q006  paraphrase     1
    q007  paraphrase     2
    q008  paraphrase     2
    q009   multi-doc     3
    q010   multi-doc     2
    q011   multi-doc     3
    q012   multi-doc     3

by class: {'literal': 4, 'multi-doc': 4, 'paraphrase': 4}
total: 12 queries · 31 rows


In [21]:
# ── q004 ──────────────────────────────────────────────────────
search("Học máy lượng tử là gì?")


,doc_id,title,chars,matched,score,snippet
0,20015836,Học máy lượng tử,4150,3,188.17,Học máy lượng tử (Quantum machine learning - Q...
1,20014464,Thuật ngữ trí tuệ nhân tạo,42756,3,147.66,"... đến ngành trí tuệ nhân tạo (AI), các phân ..."
2,24278,Học máy,19947,3,124.35,Học máy (tiếng Anh: Machine learning) là một l...
3,25735,Lịch sử ngành trí tuệ nhân tạo,97944,3,117.28,...on người kể các câu chuyện về những sinh vậ...
4,19988121,Học liên kết,37355,3,111.16,Học liên kết (tiếng Anh: federated learning) l...
5,3339820,Học sâu,57004,3,106.04,"Học sâu (tiếng Anh: deep learning, còn gọi là ..."
6,19950271,Học vấn về AI,8233,3,99.73,Học vấn về AI hay học vấn về trí tuệ nhân tạo ...
7,581452,Phép thử Turing,23683,3,89.59,...ing test) do Alan Turing đề xuất nhằm đánh ...


In [22]:
show("20015836")


# Học máy lượng tử
https://vi.wikipedia.org/wiki/H%E1%BB%8Dc_m%C3%A1y_l%C6%B0%E1%BB%A3ng_t%E1%BB%AD
(4,150 chars)

Học máy lượng tử (Quantum machine learning - QML) là lĩnh vực nghiên cứu các thuật toán lượng tử dành cho học máy. Học máy lượng tử thường đề cập đến các thuật toán lượng tử dành cho các tác vụ học máy phân tích dữ liệu cổ điển, đôi khi còn được gọi là học máy tăng cường lượng tử (Quantum-enhanced machine learning). Thuật ngữ "học máy lượng tử" đôi khi được sử dụng để chỉ các phương pháp học máy cổ điển áp dụng cho dữ liệu được tạo ra từ các thí nghiệm lượng tử (tức là học máy của các hệ lượng tử), chẳng hạn như việc học tập các Chuyển pha của một hệ lượng tử hoặc việc tạo ra các thí nghiệm lượng tử mới. Các thuật toán QML sử dụng các Qubit và các phép toán lượng tử để cố gắng cải thiện độ phức tạp không gian và độ phức tạp thời gian của các thuật toán học máy cổ điển. 
Các phương pháp QML lai (hybrid QML methods) bao gồm cả việc xử lý cổ điển và lượng tử, trong đó các chư

In [23]:
show("25735")


# Lịch sử ngành trí tuệ nhân tạo
https://vi.wikipedia.org/wiki/L%E1%BB%8Bch_s%E1%BB%AD_ng%C3%A0nh_tr%C3%AD_tu%E1%BB%87_nh%C3%A2n_t%E1%BA%A1o
(97,944 chars)

Lịch sử ngành trí tuệ nhân tạo (AI) có những tiền đề từ thời cổ đại, khi con người kể các câu chuyện về những sinh vật nhân tạo và cỗ máy được ban cho trí thông minh. Qua nhiều thế kỷ, sự phát triển của logic, toán học và các phương pháp suy luận dựa trên quy tắc đã góp phần dẫn đến sự ra đời của máy tính điện tử vào thập niên 1940. Khi máy tính có thể thực hiện nhiều chuỗi thao tác khác nhau theo chương trình, các nhà khoa học bắt đầu nghiêm túc xem xét khả năng chế tạo một "bộ não điện tử", tức một hệ thống có thể mô phỏng một số năng lực trí tuệ của con người.
Trí tuệ nhân tạo chính thức hình thành như một lĩnh vực nghiên cứu tại một hội thảo tổ chức ở Đại học Dartmouth năm 1956. Tại đây, Allen Newell và Herbert A. Simon, với sự hỗ trợ của J. C. Shaw, trình bày Logic Theorist, một trong những chương trình AI đầu tiên. Newell về 

In [24]:
show("19950271")


# Học vấn về AI
https://vi.wikipedia.org/wiki/H%E1%BB%8Dc_v%E1%BA%A5n_v%E1%BB%81_AI
(8,233 chars)

Học vấn về AI hay học vấn về trí tuệ nhân tạo (tiếng Anh: AI literacy) là khả năng hiểu, sử dụng, giám sát và suy ngẫm một cách có phê phán về các ứng dụng AI. Thuật ngữ này thường dùng để chỉ việc giảng dạy các kỹ năng và kiến thức cho công chúng, đặc biệt là những người không thành thạo về AI.
Một số người cho rằng học vấn về AI là rất cần thiết cho học sinh và sinh viên. Ngược lại, một số giáo sư lại cấm sử dụng AI trong lớp học và trong tất cả các bài tập. Họ còn đưa ra những hình phạt nghiêm khắc cho việc sử dụng AI, coi đó là một hành vi gian lận. AI được sử dụng trong nhiều ứng dụng khác nhau, ví dụ như ô tô tự lái, trợ lý ảo và các mô hình AI tạo sinh có khả năng tạo văn bản. Người dùng các công cụ này cần có khả năng đưa ra quyết định một cách sáng suốt. Học vấn về AI cũng có thể ảnh hưởng đến cơ hội việc làm trong tương lai của sinh viên.


== Định nghĩa ==
Một trong những định 

In [25]:
judge("q004", "Học máy lượng tử là gì?", {
    "20015836": 2, "19950271": 0, "25735": 0
}, "literal")

progress()


q004: +0 new rows -> 31 total
query_id query_class  docs
    q001     literal     3
    q002     literal     3
    q003     literal     3
    q004     literal     3
    q005  paraphrase     3
    q006  paraphrase     1
    q007  paraphrase     2
    q008  paraphrase     2
    q009   multi-doc     3
    q010   multi-doc     2
    q011   multi-doc     3
    q012   multi-doc     3

by class: {'literal': 4, 'multi-doc': 4, 'paraphrase': 4}
total: 12 queries · 31 rows


In [26]:
# ── persist to eval/qrels.csv ─────────────────────────────────
progress()
save()


query_id query_class  docs
    q001     literal     3
    q002     literal     3
    q003     literal     3
    q004     literal     3
    q005  paraphrase     3
    q006  paraphrase     1
    q007  paraphrase     2
    q008  paraphrase     2
    q009   multi-doc     3
    q010   multi-doc     2
    q011   multi-doc     3
    q012   multi-doc     3

by class: {'literal': 4, 'multi-doc': 4, 'paraphrase': 4}
total: 12 queries · 31 rows
saved 31 rows -> eval\qrels.csv
